# 03 — Model Comparison

Compare Logistic Regression, Random Forest, and XGBoost baselines — with and without feature selection. Uses a **50k stratified subsample** for faster iteration; full-data metrics are in notebook 04 and `outputs/metrics.json`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%matplotlib inline

import pandas as pd
from sklearn.model_selection import train_test_split

from src.config import load_config
from src.data import load_train_data
from src.evaluation import compute_metrics, evaluate_model
from src.models import (
    build_lr_pipeline,
    build_rf_pipeline,
    build_xgb_pipeline,
    compute_scale_pos_weight,
)
from src.preprocessing import prepare_xy, split_train_test

In [ ]:
config = load_config()
df = load_train_data()

SUBSAMPLE = 50_000
if len(df) > SUBSAMPLE:
    df = df.groupby("Churn", group_keys=False).apply(
        lambda g: g.sample(frac=SUBSAMPLE / len(df), random_state=config["random_state"])
    )

X, y = prepare_xy(df)
X_train, X_test, y_train, y_test = split_train_test(
    X, y,
    test_size=config["test_size"],
    random_state=config["random_state"],
)
scale_pos_weight = compute_scale_pos_weight(y_train)
print(f"Subsample size: {len(df):,} | train: {len(X_train):,} | test: {len(X_test):,}")

## Baseline models (no feature selection)

In [ ]:
results = []
random_state = config["random_state"]

baselines = {
    "Logistic Regression": build_lr_pipeline(random_state=random_state),
    "Random Forest": build_rf_pipeline(random_state=random_state),
    "XGBoost": build_xgb_pipeline(scale_pos_weight, random_state=random_state),
}

for name, pipe in baselines.items():
    pipe.fit(X_train, y_train)
    metrics = compute_metrics(
        y_test, pipe.predict(X_test), pipe.predict_proba(X_test)[:, 1]
    )
    metrics["model"] = name
    metrics["feature_selection"] = False
    results.append(metrics)
    print(f"\n=== {name} ===")
    evaluate_model(pipe, X_test, y_test, plot_confusion_matrix=False, verbose=True)

## Models with feature selection

In [ ]:
fs_models = {
    "Logistic Regression + FS": build_lr_pipeline(random_state=random_state, with_feature_selection=True),
    "Random Forest + FS": build_rf_pipeline(random_state=random_state, with_feature_selection=True),
    "XGBoost + FS": build_xgb_pipeline(
        scale_pos_weight,
        random_state=random_state,
        with_feature_selection=True,
        feature_selection_cfg=config["feature_selection"],
    ),
}

for name, pipe in fs_models.items():
    pipe.fit(X_train, y_train)
    metrics = compute_metrics(
        y_test, pipe.predict(X_test), pipe.predict_proba(X_test)[:, 1]
    )
    metrics["model"] = name
    metrics["feature_selection"] = True
    results.append(metrics)
    print(f"\n=== {name} ===")
    evaluate_model(pipe, X_test, y_test, plot_confusion_matrix=False, verbose=True)

## Comparison table

In [ ]:
comparison = pd.DataFrame(results)[
    ["model", "feature_selection", "roc_auc", "f1", "recall", "precision", "accuracy"]
].sort_values("roc_auc", ascending=False)
comparison.round(4)

## Interpretation

- **XGBoost** consistently leads on ROC-AUC, with strong recall for the minority churn class — important when false negatives (missed churners) are costly.
- Feature selection trims dimensionality with minimal AUC loss; the production pipeline uses `SelectFromModel` + tuned XGBoost.
- Full-dataset holdout ROC-AUC **~0.916** after hyperparameter tuning is documented in notebook 04.